In [1]:
import numpy as np
import pandas as pd
import sys
sys.path.append('..')
import os
import warnings
import pickle
warnings.filterwarnings("ignore")

from src.data_assemble.assemble_ml import *
from src.data_assemble.assemble_conv import *
from src.models.utils import *
random.seed(112)

In [2]:
station_names = ['Богородицкое-Фенино', 'Готня', 'Валуйки', 'Чертково', 'Цимлянск(Волгодонск)', 'Таганрог',  
                 'Гигант', 'Ремонтное', 'Приморско-Ахтарск', 'Краснодар, Круглик', 'Анапа', 'Туапсе', 'Армавир', 'Сочи',
                 'Красная Поляна']  # 'Ростов-на-Дону' 2192 instead of 3652
half_side_size = 3
# station_names = list(weatherstation_list['Наименование станции'])

In [3]:
# Automatise start and end
start = '2006-01-01'
end = '2020-12-31'
df = pd.read_csv('../data/weather_stations/data_meteo_full.csv')
target = get_y(df, start, end, station_names, speed_th=20)


In [5]:
path_to_files = '../data/CMIP/*.nc'
weatherstation_list = pd.read_csv('../data/weather_stations/weatherstation_list.csv')
# rectangle_coords = {'lat_min': 43.38, 'lat_max': 51.52,'lon_min': 35.12, 'lon_max': 44.45}
rectangle_coords = {'lat_min': 39.38, 'lat_max': 55.52,'lon_min': 30.12, 'lon_max': 49.45}
target_res = {'lon_res': 0.25, 'lat_res': 0.25}
filter_dict = {"years": ['2006', '2016'], "bands": ['Wind_', 'pr_', 'tasmax', 'tasmin']}
stations_pixs = get_pixel_stations(path_to_files, filter_dict, station_names, weatherstation_list, rectangle_coords, target_res)

100%|██████████| 3650/3650 [00:00<00:00, 6963.63it/s]


In [6]:
blocks = make_blocks([path_to_files], filter_dict, rectangle_coords, target_res, half_side_size=half_side_size)

100%|██████████| 4/4 [00:03<00:00,  1.17it/s]


In [7]:
X, y = assemble_numpy_ds(blocks, target, stations_pixs)

100%|██████████| 15/15 [00:00<00:00, 58.71it/s]


In [8]:
path_to_dump = os.path.join('..', 'data','nn_train')    # Redirect to STASH
trg_path = os.path.join(path_to_dump, 'target')
obj_path = os.path.join(path_to_dump, 'objects')
for k in X.keys():
    X_station = X[k]
    y_station = y[k]

    st_path = os.path.join(path_to_dump, k)
    if not os.path.isdir(st_path):
        os.makedirs(st_path)
    
    with open(os.path.join(st_path, 'objects.npy'),'wb') as f:
        pickle.dump(X_station, f)
        # np.save(f, X_station)
    with open(os.path.join(st_path, 'target.npy'),'wb') as f:
        pickle.dump(y_station, f)
        # np.save(f, y_station)